In [ ]:
import asyncio
from ib_insync import *
import pandas as pd
import numpy as np
from datetime import datetime
import random
from pymongo import MongoClient
from bson import ObjectId
import warnings

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# DATABASE + IB CONNECTION
# ══════════════════════════════════════════════════════════════════════════════

client               = MongoClient('mongodb://localhost:27017/')
db                   = client['dipBuy_db']
trades_collection    = db['trades']
exclude_collection   = db['excluded_tickers_v2']

util.startLoop()
ib = IB()
ib.connect('127.0.0.1', 7497, clientId=random.randint(1, 1000))

tickers = [
    'WMT', 'MU', 'SAVA', 'SLDB', 'SLV', 'NEM', 'CTXR', 'ONON', 'IONQ', 'MARA',
    'MRVL', 'T', 'CCL', 'XYZ', 'PG', 'ONDS', 'NFLX', 'V', 'ADBE', 'MS', 'CXW',
    'BA', 'BABA', 'DAL', 'JPM', 'C', 'AMZN', 'UAL', 'BRK.B', 'IBM', 'MSFT', 'KO',
    'MSTR', 'COIN', 'GLD', 'META', 'AAL', 'OSCR', 'QSI', 'SPY', 'NVO', 'DIS',
    'AAPL', 'BMNR', 'GRAB', 'RBLX', 'AFRM', 'NVDA', 'FUBO', 'PYPL', 'GOOG', 'BTC',
    'RGTI', 'GPRO', 'OKLO', 'AMD', 'XPEV', 'SHEL', 'TSM', 'SNOW', 'NET', 'SES',
    'TSLA', 'BP', 'UBER', 'INTC', 'MRNA', 'IREN', 'ORCL', 'HIMS', 'NBIS'
]
contracts = [Stock(t, 'SMART', 'USD') for t in tickers]
ib.qualifyContracts(*contracts)


# ══════════════════════════════════════════════════════════════════════════════
# INDICATORS
# ══════════════════════════════════════════════════════════════════════════════

def calc_sma(df, period):
    return df['close'].rolling(window=period).mean()

def calc_ema(df, period):
    return df['close'].ewm(span=period, adjust=False).mean()

def calc_atr(df, period=14):
    hl  = df['high'] - df['low']
    hc  = (df['high'] - df['close'].shift()).abs()
    lc  = (df['low']  - df['close'].shift()).abs()
    df['ATR'] = pd.concat([hl, hc, lc], axis=1).max(axis=1).rolling(period).mean()
    return df

def calc_rsi(df, period=14):
    d        = df['close'].diff()
    gain     = d.clip(lower=0).ewm(com=period-1, min_periods=period).mean()
    loss     = (-d.clip(upper=0)).ewm(com=period-1, min_periods=period).mean()
    df['RSI'] = 100 - 100 / (1 + gain / (loss + 1e-10))
    return df

def calc_keltner(df, ema_period=20, atr_period=14, multiplier=3.5):
    df = calc_atr(df, atr_period)
    df['KC_mid']   = df['close'].ewm(span=ema_period, adjust=False).mean()
    df['KC_upper'] = df['KC_mid'] + multiplier * df['ATR']
    df['KC_lower'] = df['KC_mid'] - multiplier * df['ATR']
    return df

def calculate_all_indicators(df):
    df = calc_keltner(df)
    df = calc_rsi(df)
    df['EMA5']   = calc_ema(df, 5)
    df['SMA200'] = calc_sma(df, 200)
    df['SMA250'] = calc_sma(df, 250)
    return df

def get_prev_day_low(df):
    """
    Lowest low of the most recently completed trading day.
    Gossett L6: sell ½ / remaining when 5-min bar closes BELOW this level.
    """
    df2          = df.copy()
    df2['d']     = pd.to_datetime(df2['date']).dt.date
    today        = df2['d'].iloc[-1]
    prior_bars   = df2[df2['d'] < today]
    if prior_bars.empty:
        return df2['low'].iloc[:-1].min()
    prev_day     = prior_bars['d'].max()
    return df2[df2['d'] == prev_day]['low'].min()


# ══════════════════════════════════════════════════════════════════════════════
# ENTRY SIGNAL — DEEP DIP BUY  (RSI + Keltner + 200/250 SMA)
# ══════════════════════════════════════════════════════════════════════════════

def detect_entry_signal(df):
    """
    ALL four conditions must be true on the latest bar:

    1. UPTREND CONTEXT  — close > SMA200 OR close > SMA250
    2. DIP TYPE         — price interacted with SMA200/250 (A/B/C)
    3. RSI RECOVERY     — RSI was ≤ 30 in last 5 bars AND is now ≥ 30
    4. KELTNER CONFIRM  — low touched KC_lower recently AND close now > KC_mid

    Returns dict with 'signal': True/False + diagnostics.
    """
    if len(df) < 10:
        return {'signal': False, 'reason': 'insufficient_data'}

    row  = df.iloc[-1]
    prev = df.iloc[-2]
    look = df.iloc[-5:]

    close    = row['close']
    low      = row['low']
    sma200   = row['SMA200']
    sma250   = row['SMA250']
    rsi_now  = row['RSI']
    rsi_prev = prev['RSI']
    kc_mid   = row['KC_mid']

    # ── 1. Uptrend context ────────────────────────────────────────
    above200 = close > sma200
    above250 = close > sma250
    if not (above200 or above250):
        return {'signal': False,
                'reason': f'below_both_sma close={close:.2f} sma200={sma200:.2f} sma250={sma250:.2f}'}

    ref_sma  = max(sma200, sma250)

    # ── 2. Dip type ───────────────────────────────────────────────
    dip_type = None
    # Type A — Reclaim: prior close below SMA, current close above
    if prev['close'] < ref_sma <= close:
        dip_type = 'A_reclaim'
    # Type B — Intraday rejection: low dipped below SMA but close is above
    elif low < ref_sma < close:
        dip_type = 'B_intraday_rejection'
    # Type C — Clean bounce: low within 0.5% of SMA and price rising
    elif abs(low - ref_sma) / ref_sma <= 0.005 and close > prev['close']:
        dip_type = 'C_clean_bounce'
    else:
        # Extended look-back (last 3 bars)
        for i in range(-3, -1):
            b = df.iloc[i]
            if b['close'] < ref_sma <= close:
                dip_type = 'A_reclaim_delayed'; break
            if b['low'] < ref_sma < close:
                dip_type = 'B_intraday_delayed'; break

    if dip_type is None:
        return {'signal': False,
                'reason': f'no_dip_type close={close:.2f} ref_sma={ref_sma:.2f}'}

    # ── 3. RSI recovery ───────────────────────────────────────────
    was_oversold = (look['RSI'] <= 30).any()
    if not (was_oversold and rsi_now >= 30):
        return {'signal': False,
                'reason': f'rsi_no_recovery rsi={rsi_now:.1f} was_oversold={was_oversold}'}

    # ── 4. Keltner confirmation ───────────────────────────────────
    kc_touched  = (look['low'] <= look['KC_lower']).any()
    kc_reclaimed = close > kc_mid
    if not kc_touched:
        return {'signal': False, 'reason': 'kc_lower_not_touched'}
    if not kc_reclaimed:
        return {'signal': False,
                'reason': f'kc_mid_not_reclaimed close={close:.2f} kc_mid={kc_mid:.2f}'}

    return {
        'signal':        True,
        'dip_type':      dip_type,
        'rsi_now':       round(rsi_now, 1),
        'was_oversold':  was_oversold,
        'cross_30':      rsi_prev < 30 <= rsi_now,
        'cross_40':      rsi_prev < 40 <= rsi_now,
        'above_sma200':  above200,
        'above_sma250':  above250,
        'close':         round(close, 2),
        'ref_sma':       round(ref_sma, 2),
        'kc_mid':        round(kc_mid, 2),
        'kc_upper':      round(row['KC_upper'], 2),
    }


# ══════════════════════════════════════════════════════════════════════════════
# MONGO HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def create_trade_doc(symbol, entry_price, qty, signal_info, atr):
    """
    Stores all Gossett L6 state alongside standard trade fields.

    Position sizing (Gossett method):
        Risk 1% of total capital ÷ (2 × ATR) = share count
        emergency_stop     = entry - 2 × ATR
        trailing_active_at = entry + 1.5 × ATR   ← trailing mode unlocks here

    Exit state flags (all start False, updated during the trade):
        half_sold        — Step 1 first-½ sell has fired
        step2a_done      — Step 2a (RSI-70 rejection sell) has fired
        rsi_was_above_70 — RSI bar closed at or above 70
        rsi_rejected_70  — RSI bar was ≥70 then closed below 70
        rsi_max          — rolling peak RSI seen while in trade
    """
    return {
        'instrument':        symbol,
        'direction':         'long',
        'entry_price':       entry_price,
        'highest_price':     entry_price,
        'quantity':          qty,
        'remaining_qty':     qty,
        'atr_at_entry':      round(atr, 4),
        'emergency_stop':    round(entry_price - 2.0 * atr, 4),   # 2 ATR hard stop
        'trailing_active_at': round(entry_price + 1.5 * atr, 4),  # 1.5 ATR unlock
        # Gossett L6 exit-state flags
        'half_sold':         False,
        'step2a_done':       False,
        'rsi_was_above_70':  False,
        'rsi_rejected_70':   False,
        'rsi_max':           0.0,
        # Signal snapshot
        'signal_info':       signal_info,
        'entry_timestamp':   datetime.now(),
        # Standard fields
        'order_id':          str(ObjectId()),
        'exit_price':        None,
        'exit_timestamp':    None,
        'reason':            None,
        'current_pnl':       0,
        'realized_pnl':      0,
        'status':            'open',
        'created_at':        datetime.now(),
        'updated_at':        datetime.now(),
    }

def db_update(trade_id, updates):
    updates['updated_at'] = datetime.now()
    trades_collection.update_one({'_id': ObjectId(trade_id)}, {'$set': updates})

def do_sell(contract, qty, symbol, entry_price, sell_price, reason, rsi, trade_id, status='closed'):
    """Execute market sell, persist exit to MongoDB, print summary."""
    ib.placeOrder(contract, MarketOrder('SELL', qty))
    pnl = (sell_price - entry_price) * qty
    db_update(trade_id, {
        'exit_price':     sell_price,
        'exit_timestamp': datetime.now(),
        'reason':         reason,
        'realized_pnl':   pnl,
        'exit_rsi':       rsi,
        'status':         status,
    })
    sign = '+' if pnl >= 0 else ''
    print(f"  ✅ SELL {qty}sh @ ${sell_price:.2f}  [{reason}]  PnL: {sign}${pnl:.2f}")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN LOOP
# ══════════════════════════════════════════════════════════════════════════════

async def check_and_trade():
    """
    ╔══════════════════════════════════════════════════════════════════════════╗
    ║  GOSSETT LESSON 6 — COMPLETE EXIT RULE SET                              ║
    ╠══════════════════════════════════════════════════════════════════════════╣
    ║                                                                          ║
    ║  HARD STOPS  (checked first on every bar, always active):               ║
    ║                                                                          ║
    ║    • EMERGENCY STOP  = entry price − 2 × ATR                            ║
    ║      → EXIT ALL immediately if current price ≤ emergency_stop           ║
    ║      → Represents ~1% of total capital risk                             ║
    ║                                                                          ║
    ║    • END-OF-DAY STOP = close below SMA 200                              ║
    ║      → EXIT ALL remaining shares                                         ║
    ║                                                                          ║
    ║  TRAILING MODE  (unlocks once price rises 1.5 ATR above entry):         ║
    ║                                                                          ║
    ║    STEP 1  ─ Sell FIRST ½ of position                                   ║
    ║      Trigger: bar closes below the PREVIOUS DAY'S LOW                   ║
    ║                                                                          ║
    ║    STEP 2a ─ Sell ½ of REMAINING shares  [RSI 70 path]                 ║
    ║      Trigger: RSI closed ≥ 70 on any prior bar (rsi_was_above_70)       ║
    ║               AND current bar RSI closes back BELOW 70                  ║
    ║               (intra-day spike above 70 then close below also counts)   ║
    ║                                                                          ║
    ║    STEP 3  ─ Sell ALL remaining shares  [after Step 2a]                 ║
    ║      Trigger: next bar that closes below the PREVIOUS DAY'S LOW         ║
    ║                                                                          ║
    ║    STEP 2b ─ EMA 5 FALLBACK  [if RSI peaked 65–69, never hit 70]       ║
    ║      Trigger: rsi_max ≥ 65 AND rsi never reached 70                     ║
    ║               AND price closes below EMA 5                              ║
    ║      → EXIT ALL remaining shares                                         ║
    ║                                                                          ║
    ╚══════════════════════════════════════════════════════════════════════════╝
    """

    while True:
        ib.positions()   # refresh internal positions cache

        for contract in contracts:
            symbol = contract.symbol
            print(f"\n{'='*60}")
            print(f"  {symbol}  {datetime.now().strftime('%H:%M:%S')}")
            print(f"{'='*60}")

            # ── Fetch 5-min bars (3 days for SMA 200/250 warmup) ──────────
            bars = ib.reqHistoricalData(
                contract,
                endDateTime='',
                durationStr='3 D',
                barSizeSetting='5 mins',
                whatToShow='TRADES',
                useRTH=False,
                formatDate=1
            )
            if not bars:
                print(f"  No data for {symbol}")
                continue

            df            = util.df(bars)
            df.columns    = [c.lower() for c in df.columns]
            df            = calculate_all_indicators(df)

            row           = df.iloc[-1]
            prev_row      = df.iloc[-2]
            price         = row['close']
            atr           = max(float(row['ATR']), 0.01)
            rsi_now       = float(row['RSI'])
            rsi_prev      = float(prev_row['RSI'])
            ema5          = float(row['EMA5'])
            sma200        = float(row['SMA200'])
            prev_day_low  = get_prev_day_low(df)

            print(f"  Price=${price:.2f}  RSI={rsi_now:.1f}  EMA5=${ema5:.2f}"
                  f"  SMA200=${sma200:.2f}  PrevDayLow=${prev_day_low:.2f}")

            open_trade = trades_collection.find_one({'instrument': symbol, 'status': 'open'})

            # ══════════════════════════════════════════════════════════════
            # POSITION MANAGEMENT — EXIT LOGIC
            # ══════════════════════════════════════════════════════════════

            if open_trade:
                entry_price        = float(open_trade['entry_price'])
                orig_qty           = int(open_trade['quantity'])
                rem_qty            = int(open_trade.get('remaining_qty', orig_qty))
                half_sold          = bool(open_trade.get('half_sold', False))
                step2a_done        = bool(open_trade.get('step2a_done', False))
                rsi_was_above_70   = bool(open_trade.get('rsi_was_above_70', False))
                rsi_rejected_70    = bool(open_trade.get('rsi_rejected_70', False))
                rsi_max            = float(open_trade.get('rsi_max', 0.0))
                emergency_stop     = float(open_trade['emergency_stop'])
                trailing_active_at = float(open_trade['trailing_active_at'])
                highest_price      = float(open_trade.get('highest_price', entry_price))
                trade_id           = open_trade['_id']

                profit_pct  = (price - entry_price) / entry_price
                current_pnl = (price - entry_price) * rem_qty

                # ── Update rolling state trackers ─────────────────────────
                state_upd = {'current_pnl': current_pnl}

                if price > highest_price:
                    highest_price = price
                    state_upd['highest_price'] = highest_price

                if rsi_now > rsi_max:
                    rsi_max = rsi_now
                    state_upd['rsi_max'] = rsi_max

                # RSI crossed above 70 — flag it
                if rsi_now >= 70 and not rsi_was_above_70:
                    rsi_was_above_70 = True
                    state_upd['rsi_was_above_70'] = True
                    print(f"  📌 RSI crossed above 70 ({rsi_now:.1f}) — tracking rejection")

                # RSI rejection: was ≥70 on previous bar, now closed below 70
                if rsi_was_above_70 and not rsi_rejected_70 and rsi_prev >= 70 and rsi_now < 70:
                    rsi_rejected_70 = True
                    state_upd['rsi_rejected_70'] = True
                    print(f"  📌 RSI 70 REJECTION confirmed (prev={rsi_prev:.1f} → now={rsi_now:.1f})")

                db_update(trade_id, state_upd)

                trailing_mode = (highest_price >= trailing_active_at)

                print(f"  OPEN  entry=${entry_price:.2f}  rem={rem_qty}/{orig_qty}"
                      f"  P&L={profit_pct:+.2%}"
                      f"  trailing={'ON' if trailing_mode else 'OFF'}"
                      f"  emerg_stop=${emergency_stop:.2f}"
                      f"  trail_at=${trailing_active_at:.2f}")
                print(f"  half_sold={half_sold}  step2a={step2a_done}"
                      f"  rsi_was70={rsi_was_above_70}  rsi_rej70={rsi_rejected_70}"
                      f"  rsi_max={rsi_max:.1f}")

                # ──────────────────────────────────────────────────────────
                # HARD STOP A — Emergency: close ≤ entry - 2 ATR
                # Gossett: "exit immediately for a 1% loss of total capital"
                # ──────────────────────────────────────────────────────────
                if price <= emergency_stop and rem_qty > 0:
                    do_sell(contract, rem_qty, symbol, entry_price, price,
                            f'EMERGENCY_STOP_2ATR_@{emergency_stop:.2f}',
                            rsi_now, trade_id, status='closed')
                    print(f"  🚨 EMERGENCY STOP hit @ ${price:.2f}"
                          f" (entry ${entry_price:.2f} − 2×ATR {2*float(open_trade['atr_at_entry']):.2f})")
                    continue

                # ──────────────────────────────────────────────────────────
                # HARD STOP B — End-of-day: close below SMA 200
                # Gossett: "exit all remaining shares"
                # ──────────────────────────────────────────────────────────
                if price < sma200 and rem_qty > 0:
                    do_sell(contract, rem_qty, symbol, entry_price, price,
                            f'EOD_STOP_BELOW_SMA200_@{sma200:.2f}',
                            rsi_now, trade_id, status='closed')
                    print(f"  🚨 END-OF-DAY STOP: close ${price:.2f} < SMA200 ${sma200:.2f}")
                    continue

                # ──────────────────────────────────────────────────────────
                # TRAILING MODE EXITS
                # (only active once price has risen 1.5 ATR above entry)
                # ──────────────────────────────────────────────────────────
                if trailing_mode and rem_qty > 0:

                    half_qty = max(1, orig_qty // 2)

                    # ── STEP 1 — Sell FIRST ½ ─────────────────────────────
                    # Trigger: close below previous day's LOW
                    # Gossett: "sell ½ of your position when any subsequent
                    #           price bars close below the previous day's LOW"
                    if not half_sold and price < prev_day_low:
                        sell_qty = min(half_qty, rem_qty)
                        new_rem  = rem_qty - sell_qty
                        do_sell(contract, sell_qty, symbol, entry_price, price,
                                f'STEP1_FIRST_HALF_BELOW_PREV_DAY_LOW_{prev_day_low:.2f}',
                                rsi_now, trade_id,
                                status='partial' if new_rem > 0 else 'closed')
                        db_update(trade_id, {'half_sold': True, 'remaining_qty': new_rem})
                        half_sold = True
                        rem_qty   = new_rem
                        print(f"  📉 STEP 1: sold {sell_qty}sh. Remaining: {rem_qty}")
                        if rem_qty <= 0:
                            continue

                    # ── STEP 2a — Sell ½ of REMAINING on RSI 70 rejection ─
                    # Trigger: RSI was ≥70 and has now closed back below 70
                    # Gossett: "sell another ½ of shares remaining once RSI
                    #           closes above 70 and subsequently closes back below"
                    # Note: fire only once (step2a_done flag)
                    if half_sold and not step2a_done and rsi_rejected_70 and rem_qty > 0:
                        sell_qty = max(1, rem_qty // 2)
                        new_rem  = rem_qty - sell_qty
                        do_sell(contract, sell_qty, symbol, entry_price, price,
                                f'STEP2a_HALF_REMAINING_RSI70_REJECTION_rsi={rsi_now:.1f}',
                                rsi_now, trade_id,
                                status='partial' if new_rem > 0 else 'closed')
                        db_update(trade_id, {'step2a_done': True, 'remaining_qty': new_rem})
                        step2a_done = True
                        rem_qty     = new_rem
                        print(f"  📉 STEP 2a: sold {sell_qty}sh (RSI 70 rejection). Remaining: {rem_qty}")
                        if rem_qty <= 0:
                            continue

                    # ── STEP 3 — Sell ALL remaining after Step 2a ─────────
                    # Trigger: NEXT close below previous day's LOW
                    # Gossett: "sell all remaining shares after the 70 RSI
                    #           rejection when any subsequent bars close below
                    #           the previous day's LOW"
                    if half_sold and step2a_done and price < prev_day_low and rem_qty > 0:
                        do_sell(contract, rem_qty, symbol, entry_price, price,
                                f'STEP3_FINAL_REMAINING_BELOW_PREV_DAY_LOW_{prev_day_low:.2f}',
                                rsi_now, trade_id, status='closed')
                        print(f"  📉 STEP 3: sold all {rem_qty}sh remaining "
                              f"(close below prev day low after RSI 70 rejection)")
                        continue

                    # ── STEP 2b — EMA 5 FALLBACK ──────────────────────────
                    # Used when RSI got close (65–69) but NEVER reached 70.
                    # Gossett: "exit entire position if you get a close below
                    #           the 5 EMA — implemented if RSI gets to at least 65"
                    ema5_fallback = (rsi_max >= 65 and not rsi_was_above_70)
                    if half_sold and ema5_fallback and price < ema5 and rem_qty > 0:
                        do_sell(contract, rem_qty, symbol, entry_price, price,
                                f'STEP2b_EMA5_FALLBACK_rsi_max={rsi_max:.1f}',
                                rsi_now, trade_id, status='closed')
                        print(f"  📉 STEP 2b EMA5 fallback: sold {rem_qty}sh "
                              f"(RSI peaked {rsi_max:.1f}, never hit 70, "
                              f"close ${price:.2f} < EMA5 ${ema5:.2f})")
                        continue

                else:
                    if not trailing_mode:
                        gap = trailing_active_at - price
                        print(f"  Waiting for trailing mode: "
                              f"need +${gap:.2f} to reach ${trailing_active_at:.2f} "
                              f"(entry + 1.5×ATR)")

            # ══════════════════════════════════════════════════════════════
            # ENTRY LOGIC — DEEP DIP BUY
            # ══════════════════════════════════════════════════════════════

            else:
                today = datetime.now().date()
                if exclude_collection.find_one({'ticker': symbol, 'date': today.isoformat()}):
                    print(f"  Already traded {symbol} today — skipping.")
                    continue

                signal = detect_entry_signal(df)

                if signal['signal']:
                    # ── Position size: $1000 risk ÷ (2 × ATR) ───────────
                    # Gossett: risk 1% of total capital ($1,000 on $100k account)
                    # Fixed position size: 40 shares
                    order_size   = 40

                    ib.placeOrder(contract, MarketOrder('BUY', order_size))

                    doc = create_trade_doc(symbol, price, order_size, signal, atr)
                    trades_collection.insert_one(doc)
                    exclude_collection.insert_one({'ticker': symbol, 'date': today.isoformat()})

                    print(f"  🚀 DEEP DIP BUY  {symbol}")
                    print(f"     Entry:           ${price:.2f}")
                    print(f"     Shares:          {order_size}  (fixed position size)")
                    print(f"     Dip Type:        {signal['dip_type']}")
                    print(f"     RSI:             {signal['rsi_now']}  (was_oversold={signal['was_oversold']})")
                    print(f"     KC mid:         ${signal['kc_mid']:.2f}  (reclaimed ✓)")
                    print(f"     Emergency stop: ${price - 2*atr:.2f}  (entry − 2×ATR)")
                    print(f"     Trailing unlocks:${price + 1.5*atr:.2f}  (entry + 1.5×ATR)")
                    print(f"     KC Upper target:${signal['kc_upper']:.2f}")
                else:
                    print(f"  No entry — {signal.get('reason', 'n/a')}")

            await asyncio.sleep(0.5)   # brief pause between tickers

        print(f"\n{'='*60}")
        print(f"  Scan complete. Next scan in 5 minutes.")
        print(f"{'='*60}")
        await asyncio.sleep(300)


# ══════════════════════════════════════════════════════════════════════════════
# RUN
# ══════════════════════════════════════════════════════════════════════════════

try:
    ib.run(check_and_trade())
except Exception as e:
    print(f"Error: {e}")
    import traceback; traceback.print_exc()
finally:
    ib.disconnect()
    print("Disconnected from IB.")


In [ ]:
import asyncio
from ib_insync import *
import pandas as pd
import numpy as np
from datetime import datetime
import random
from pymongo import MongoClient
from bson import ObjectId
import warnings

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# DATABASE + IB CONNECTION
# ══════════════════════════════════════════════════════════════════════════════

client               = MongoClient('mongodb://localhost:27017/')
db                   = client['dipBuy_db']
trades_collection    = db['trades']
exclude_collection   = db['excluded_tickers_v2']

util.startLoop()
ib = IB()
ib.connect('127.0.0.1', 7497, clientId=random.randint(1, 1000))

tickers = [
    'WMT', 'MU', 'SAVA', 'SLDB', 'SLV', 'NEM', 'CTXR', 'ONON', 'IONQ', 'MARA',
    'MRVL', 'T', 'CCL', 'XYZ', 'PG', 'ONDS', 'NFLX', 'V', 'ADBE', 'MS', 'CXW',
    'BA', 'BABA', 'DAL', 'JPM', 'C', 'AMZN', 'UAL', 'BRK.B', 'IBM', 'MSFT', 'KO',
    'MSTR', 'COIN', 'GLD', 'META', 'AAL', 'OSCR', 'QSI', 'SPY', 'NVO', 'DIS',
    'AAPL', 'BMNR', 'GRAB', 'RBLX', 'AFRM', 'NVDA', 'FUBO', 'PYPL', 'GOOG', 'BTC',
    'RGTI', 'GPRO', 'OKLO', 'AMD', 'XPEV', 'SHEL', 'TSM', 'SNOW', 'NET', 'SES',
    'TSLA', 'BP', 'UBER', 'INTC', 'MRNA', 'IREN', 'ORCL', 'HIMS', 'NBIS'
]
contracts = [Stock(t, 'SMART', 'USD') for t in tickers]
ib.qualifyContracts(*contracts)


# ══════════════════════════════════════════════════════════════════════════════
# INDICATORS
# ══════════════════════════════════════════════════════════════════════════════

def calc_sma(df, period):
    return df['close'].rolling(window=period).mean()

def calc_ema(df, period):
    return df['close'].ewm(span=period, adjust=False).mean()

def calc_atr(df, period=14):
    hl  = df['high'] - df['low']
    hc  = (df['high'] - df['close'].shift()).abs()
    lc  = (df['low']  - df['close'].shift()).abs()
    df['ATR'] = pd.concat([hl, hc, lc], axis=1).max(axis=1).rolling(period).mean()
    return df

def calc_rsi(df, period=14):
    d        = df['close'].diff()
    gain     = d.clip(lower=0).ewm(com=period-1, min_periods=period).mean()
    loss     = (-d.clip(upper=0)).ewm(com=period-1, min_periods=period).mean()
    df['RSI'] = 100 - 100 / (1 + gain / (loss + 1e-10))
    return df

def calc_keltner(df, ema_period=20, atr_period=14, multiplier=3.5):
    df = calc_atr(df, atr_period)
    df['KC_mid']   = df['close'].ewm(span=ema_period, adjust=False).mean()
    df['KC_upper'] = df['KC_mid'] + multiplier * df['ATR']
    df['KC_lower'] = df['KC_mid'] - multiplier * df['ATR']
    return df

def calculate_all_indicators(df):
    df = calc_keltner(df)
    df = calc_rsi(df)
    df['EMA5']   = calc_ema(df, 5)
    df['SMA200'] = calc_sma(df, 200)
    df['SMA250'] = calc_sma(df, 250)
    return df

def get_prev_day_low(df):
    """
    Lowest low of the most recently completed trading day.
    Gossett L6: sell ½ / remaining when 5-min bar closes BELOW this level.
    """
    df2          = df.copy()
    df2['d']     = pd.to_datetime(df2['date']).dt.date
    today        = df2['d'].iloc[-1]
    prior_bars   = df2[df2['d'] < today]
    if prior_bars.empty:
        return df2['low'].iloc[:-1].min()
    prev_day     = prior_bars['d'].max()
    return df2[df2['d'] == prev_day]['low'].min()


# ══════════════════════════════════════════════════════════════════════════════
# ENTRY SIGNAL — DEEP DIP BUY  (RSI + Keltner only)
# ══════════════════════════════════════════════════════════════════════════════

def detect_entry_signal(df):
    """
    TWO conditions must be true on the latest bar:

    1. RSI RECOVERY     — RSI was ≤ 30 in last 5 bars AND is now ≥ 30
    2. KELTNER CONFIRM  — low touched KC_lower recently AND close now > KC_mid

    Returns dict with 'signal': True/False + diagnostics.
    """
    if len(df) < 10:
        return {'signal': False, 'reason': 'insufficient_data'}

    row  = df.iloc[-1]
    prev = df.iloc[-2]
    look = df.iloc[-5:]

    close    = row['close']
    rsi_now  = row['RSI']
    rsi_prev = prev['RSI']
    kc_mid   = row['KC_mid']

    # ── 1. RSI recovery ───────────────────────────────────────────
    was_oversold = (look['RSI'] <= 30).any()
    if not (was_oversold and rsi_now >= 30):
        return {'signal': False,
                'reason': f'rsi_no_recovery rsi={rsi_now:.1f} was_oversold={was_oversold}'}

    # ── 2. Keltner confirmation ───────────────────────────────────
    kc_touched   = (look['low'] <= look['KC_lower']).any()
    kc_reclaimed = close > kc_mid
    if not kc_touched:
        return {'signal': False, 'reason': 'kc_lower_not_touched'}
    if not kc_reclaimed:
        return {'signal': False,
                'reason': f'kc_mid_not_reclaimed close={close:.2f} kc_mid={kc_mid:.2f}'}

    return {
        'signal':       True,
        'rsi_now':      round(rsi_now, 1),
        'was_oversold': was_oversold,
        'cross_30':     rsi_prev < 30 <= rsi_now,
        'cross_40':     rsi_prev < 40 <= rsi_now,
        'close':        round(close, 2),
        'kc_mid':       round(kc_mid, 2),
        'kc_upper':     round(row['KC_upper'], 2),
    }


# ══════════════════════════════════════════════════════════════════════════════
# MONGO HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def create_trade_doc(symbol, entry_price, qty, signal_info, atr):
    return {
        'instrument':         symbol,
        'direction':          'long',
        'entry_price':        entry_price,
        'highest_price':      entry_price,
        'quantity':           qty,
        'remaining_qty':      qty,
        'atr_at_entry':       round(atr, 4),
        'emergency_stop':     round(entry_price - 2.0 * atr, 4),
        'trailing_active_at': round(entry_price + 1.5 * atr, 4),
        # Gossett L6 exit-state flags
        'half_sold':          False,
        'step2a_done':        False,
        'rsi_was_above_70':   False,
        'rsi_rejected_70':    False,
        'rsi_max':            0.0,
        # Signal snapshot
        'signal_info':        signal_info,
        'entry_timestamp':    datetime.now(),
        # Standard fields
        'order_id':           str(ObjectId()),
        'exit_price':         None,
        'exit_timestamp':     None,
        'reason':             None,
        'current_pnl':        0,
        'realized_pnl':       0,
        'status':             'open',
        'created_at':         datetime.now(),
        'updated_at':         datetime.now(),
    }

def db_update(trade_id, updates):
    updates['updated_at'] = datetime.now()
    trades_collection.update_one({'_id': ObjectId(trade_id)}, {'$set': updates})

def do_sell(contract, qty, symbol, entry_price, sell_price, reason, rsi, trade_id, status='closed'):
    """Execute market sell, persist exit to MongoDB, print summary."""
    ib.placeOrder(contract, MarketOrder('SELL', qty))
    pnl = (sell_price - entry_price) * qty
    db_update(trade_id, {
        'exit_price':     sell_price,
        'exit_timestamp': datetime.now(),
        'reason':         reason,
        'realized_pnl':   pnl,
        'exit_rsi':       rsi,
        'status':         status,
    })
    sign = '+' if pnl >= 0 else ''
    print(f"  ✅ SELL {qty}sh @ ${sell_price:.2f}  [{reason}]  PnL: {sign}${pnl:.2f}")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN LOOP
# ══════════════════════════════════════════════════════════════════════════════

async def check_and_trade():
    """
    ╔══════════════════════════════════════════════════════════════════════════╗
    ║  GOSSETT LESSON 6 — COMPLETE EXIT RULE SET                              ║
    ╠══════════════════════════════════════════════════════════════════════════╣
    ║                                                                          ║
    ║  HARD STOPS  (checked first on every bar, always active):               ║
    ║                                                                          ║
    ║    • EMERGENCY STOP  = entry price − 2 × ATR                            ║
    ║      → EXIT ALL immediately if current price ≤ emergency_stop           ║
    ║                                                                          ║
    ║    • END-OF-DAY STOP = close below SMA 200                              ║
    ║      → EXIT ALL remaining shares                                         ║
    ║                                                                          ║
    ║  TRAILING MODE  (unlocks once price rises 1.5 ATR above entry):         ║
    ║                                                                          ║
    ║    STEP 1  ─ Sell FIRST ½ of position                                   ║
    ║      Trigger: bar closes below the PREVIOUS DAY'S LOW                   ║
    ║                                                                          ║
    ║    STEP 2a ─ Sell ½ of REMAINING shares  [RSI 70 path]                 ║
    ║      Trigger: RSI closed ≥ 70 on any prior bar AND current RSI < 70    ║
    ║                                                                          ║
    ║    STEP 3  ─ Sell ALL remaining shares  [after Step 2a]                 ║
    ║      Trigger: next bar that closes below the PREVIOUS DAY'S LOW         ║
    ║                                                                          ║
    ║    STEP 2b ─ EMA 5 FALLBACK  [if RSI peaked 65–69, never hit 70]       ║
    ║      Trigger: rsi_max ≥ 65 AND rsi never reached 70                     ║
    ║               AND price closes below EMA 5                              ║
    ║      → EXIT ALL remaining shares                                         ║
    ║                                                                          ║
    ╚══════════════════════════════════════════════════════════════════════════╝
    """

    while True:
        ib.positions()   # refresh internal positions cache

        for contract in contracts:
            symbol = contract.symbol
            print(f"\n{'='*60}")
            print(f"  {symbol}  {datetime.now().strftime('%H:%M:%S')}")
            print(f"{'='*60}")

            # ── Fetch 5-min bars (3 days for SMA 200/250 warmup) ──────────
            bars = ib.reqHistoricalData(
                contract,
                endDateTime='',
                durationStr='3 D',
                barSizeSetting='5 mins',
                whatToShow='TRADES',
                useRTH=False,
                formatDate=1
            )
            if not bars:
                print(f"  No data for {symbol}")
                continue

            df            = util.df(bars)
            df.columns    = [c.lower() for c in df.columns]
            df            = calculate_all_indicators(df)

            row           = df.iloc[-1]
            prev_row      = df.iloc[-2]
            price         = row['close']
            atr           = max(float(row['ATR']), 0.01)
            rsi_now       = float(row['RSI'])
            rsi_prev      = float(prev_row['RSI'])
            ema5          = float(row['EMA5'])
            sma200        = float(row['SMA200'])
            prev_day_low  = get_prev_day_low(df)

            print(f"  Price=${price:.2f}  RSI={rsi_now:.1f}  EMA5=${ema5:.2f}"
                  f"  SMA200=${sma200:.2f}  PrevDayLow=${prev_day_low:.2f}")

            open_trade = trades_collection.find_one({'instrument': symbol, 'status': 'open'})

            # ══════════════════════════════════════════════════════════════
            # POSITION MANAGEMENT — EXIT LOGIC
            # ══════════════════════════════════════════════════════════════

            if open_trade:
                entry_price        = float(open_trade['entry_price'])
                orig_qty           = int(open_trade['quantity'])
                rem_qty            = int(open_trade.get('remaining_qty', orig_qty))
                half_sold          = bool(open_trade.get('half_sold', False))
                step2a_done        = bool(open_trade.get('step2a_done', False))
                rsi_was_above_70   = bool(open_trade.get('rsi_was_above_70', False))
                rsi_rejected_70    = bool(open_trade.get('rsi_rejected_70', False))
                rsi_max            = float(open_trade.get('rsi_max', 0.0))
                emergency_stop     = float(open_trade['emergency_stop'])
                trailing_active_at = float(open_trade['trailing_active_at'])
                highest_price      = float(open_trade.get('highest_price', entry_price))
                trade_id           = open_trade['_id']

                profit_pct  = (price - entry_price) / entry_price
                current_pnl = (price - entry_price) * rem_qty

                # ── Update rolling state trackers ─────────────────────────
                state_upd = {'current_pnl': current_pnl}

                if price > highest_price:
                    highest_price = price
                    state_upd['highest_price'] = highest_price

                if rsi_now > rsi_max:
                    rsi_max = rsi_now
                    state_upd['rsi_max'] = rsi_max

                # RSI crossed above 70 — flag it
                if rsi_now >= 70 and not rsi_was_above_70:
                    rsi_was_above_70 = True
                    state_upd['rsi_was_above_70'] = True
                    print(f"  📌 RSI crossed above 70 ({rsi_now:.1f}) — tracking rejection")

                # RSI rejection: was ≥70 on previous bar, now closed below 70
                if rsi_was_above_70 and not rsi_rejected_70 and rsi_prev >= 70 and rsi_now < 70:
                    rsi_rejected_70 = True
                    state_upd['rsi_rejected_70'] = True
                    print(f"  📌 RSI 70 REJECTION confirmed (prev={rsi_prev:.1f} → now={rsi_now:.1f})")

                db_update(trade_id, state_upd)

                trailing_mode = (highest_price >= trailing_active_at)

                print(f"  OPEN  entry=${entry_price:.2f}  rem={rem_qty}/{orig_qty}"
                      f"  P&L={profit_pct:+.2%}"
                      f"  trailing={'ON' if trailing_mode else 'OFF'}"
                      f"  emerg_stop=${emergency_stop:.2f}"
                      f"  trail_at=${trailing_active_at:.2f}")
                print(f"  half_sold={half_sold}  step2a={step2a_done}"
                      f"  rsi_was70={rsi_was_above_70}  rsi_rej70={rsi_rejected_70}"
                      f"  rsi_max={rsi_max:.1f}")

                # ──────────────────────────────────────────────────────────
                # HARD STOP A — Emergency: close ≤ entry - 2 ATR
                # ──────────────────────────────────────────────────────────
                if price <= emergency_stop and rem_qty > 0:
                    do_sell(contract, rem_qty, symbol, entry_price, price,
                            f'EMERGENCY_STOP_2ATR_@{emergency_stop:.2f}',
                            rsi_now, trade_id, status='closed')
                    print(f"  🚨 EMERGENCY STOP hit @ ${price:.2f}"
                          f" (entry ${entry_price:.2f} − 2×ATR {2*float(open_trade['atr_at_entry']):.2f})")
                    continue

                # ──────────────────────────────────────────────────────────
                # HARD STOP B — End-of-day: close below SMA 200
                # ──────────────────────────────────────────────────────────
                if price < sma200 and rem_qty > 0:
                    do_sell(contract, rem_qty, symbol, entry_price, price,
                            f'EOD_STOP_BELOW_SMA200_@{sma200:.2f}',
                            rsi_now, trade_id, status='closed')
                    print(f"  🚨 END-OF-DAY STOP: close ${price:.2f} < SMA200 ${sma200:.2f}")
                    continue

                # ──────────────────────────────────────────────────────────
                # TRAILING MODE EXITS
                # ──────────────────────────────────────────────────────────
                if trailing_mode and rem_qty > 0:

                    half_qty = max(1, orig_qty // 2)

                    # ── STEP 1 — Sell FIRST ½ ─────────────────────────────
                    if not half_sold and price < prev_day_low:
                        sell_qty = min(half_qty, rem_qty)
                        new_rem  = rem_qty - sell_qty
                        do_sell(contract, sell_qty, symbol, entry_price, price,
                                f'STEP1_FIRST_HALF_BELOW_PREV_DAY_LOW_{prev_day_low:.2f}',
                                rsi_now, trade_id,
                                status='partial' if new_rem > 0 else 'closed')
                        db_update(trade_id, {'half_sold': True, 'remaining_qty': new_rem})
                        half_sold = True
                        rem_qty   = new_rem
                        print(f"  📉 STEP 1: sold {sell_qty}sh. Remaining: {rem_qty}")
                        if rem_qty <= 0:
                            continue

                    # ── STEP 2a — Sell ½ of REMAINING on RSI 70 rejection ─
                    if half_sold and not step2a_done and rsi_rejected_70 and rem_qty > 0:
                        sell_qty = max(1, rem_qty // 2)
                        new_rem  = rem_qty - sell_qty
                        do_sell(contract, sell_qty, symbol, entry_price, price,
                                f'STEP2a_HALF_REMAINING_RSI70_REJECTION_rsi={rsi_now:.1f}',
                                rsi_now, trade_id,
                                status='partial' if new_rem > 0 else 'closed')
                        db_update(trade_id, {'step2a_done': True, 'remaining_qty': new_rem})
                        step2a_done = True
                        rem_qty     = new_rem
                        print(f"  📉 STEP 2a: sold {sell_qty}sh (RSI 70 rejection). Remaining: {rem_qty}")
                        if rem_qty <= 0:
                            continue

                    # ── STEP 3 — Sell ALL remaining after Step 2a ─────────
                    if half_sold and step2a_done and price < prev_day_low and rem_qty > 0:
                        do_sell(contract, rem_qty, symbol, entry_price, price,
                                f'STEP3_FINAL_REMAINING_BELOW_PREV_DAY_LOW_{prev_day_low:.2f}',
                                rsi_now, trade_id, status='closed')
                        print(f"  📉 STEP 3: sold all {rem_qty}sh remaining "
                              f"(close below prev day low after RSI 70 rejection)")
                        continue

                    # ── STEP 2b — EMA 5 FALLBACK ──────────────────────────
                    ema5_fallback = (rsi_max >= 65 and not rsi_was_above_70)
                    if half_sold and ema5_fallback and price < ema5 and rem_qty > 0:
                        do_sell(contract, rem_qty, symbol, entry_price, price,
                                f'STEP2b_EMA5_FALLBACK_rsi_max={rsi_max:.1f}',
                                rsi_now, trade_id, status='closed')
                        print(f"  📉 STEP 2b EMA5 fallback: sold {rem_qty}sh "
                              f"(RSI peaked {rsi_max:.1f}, never hit 70, "
                              f"close ${price:.2f} < EMA5 ${ema5:.2f})")
                        continue

                else:
                    if not trailing_mode:
                        gap = trailing_active_at - price
                        print(f"  Waiting for trailing mode: "
                              f"need +${gap:.2f} to reach ${trailing_active_at:.2f} "
                              f"(entry + 1.5×ATR)")

            # ══════════════════════════════════════════════════════════════
            # ENTRY LOGIC — DEEP DIP BUY
            # ══════════════════════════════════════════════════════════════

            else:
                today = datetime.now().date()
                if exclude_collection.find_one({'ticker': symbol, 'date': today.isoformat()}):
                    print(f"  Already traded {symbol} today — skipping.")
                    continue

                signal = detect_entry_signal(df)

                if signal['signal']:
                    order_size = 40

                    ib.placeOrder(contract, MarketOrder('BUY', order_size))

                    doc = create_trade_doc(symbol, price, order_size, signal, atr)
                    trades_collection.insert_one(doc)
                    exclude_collection.insert_one({'ticker': symbol, 'date': today.isoformat()})

                    print(f"  🚀 DEEP DIP BUY  {symbol}")
                    print(f"     Entry:            ${price:.2f}")
                    print(f"     Shares:           {order_size}  (fixed position size)")
                    print(f"     RSI:              {signal['rsi_now']}  (was_oversold={signal['was_oversold']})")
                    print(f"     KC mid:          ${signal['kc_mid']:.2f}  (reclaimed ✓)")
                    print(f"     Emergency stop:  ${price - 2*atr:.2f}  (entry − 2×ATR)")
                    print(f"     Trailing unlocks: ${price + 1.5*atr:.2f}  (entry + 1.5×ATR)")
                    print(f"     KC Upper target: ${signal['kc_upper']:.2f}")
                else:
                    print(f"  No entry — {signal.get('reason', 'n/a')}")

            await asyncio.sleep(0.5)   # brief pause between tickers

        print(f"\n{'='*60}")
        print(f"  Scan complete. Next scan in 5 minutes.")
        print(f"{'='*60}")
        await asyncio.sleep(300)


# ══════════════════════════════════════════════════════════════════════════════
# RUN
# ══════════════════════════════════════════════════════════════════════════════

try:
    ib.run(check_and_trade())
except Exception as e:
    print(f"Error: {e}")
    import traceback; traceback.print_exc()
finally:
    ib.disconnect()
    print("Disconnected from IB.")

In [ ]:
import asyncio
from ib_insync import *
import pandas as pd
import numpy as np
from datetime import datetime
import random
from pymongo import MongoClient
from bson import ObjectId
import warnings

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# DATABASE + IB CONNECTION
# ══════════════════════════════════════════════════════════════════════════════

client               = MongoClient('mongodb://localhost:27017/')
db                   = client['dipBuy_db']
trades_collection    = db['trades']
exclude_collection   = db['excluded_tickers_v2']

util.startLoop()
ib = IB()
ib.connect('127.0.0.1', 7497, clientId=random.randint(1, 1000))

tickers = [
    'WMT', 'MU', 'SAVA', 'SLDB', 'SLV', 'NEM', 'CTXR', 'ONON', 'IONQ', 'MARA',
    'MRVL', 'T', 'CCL', 'XYZ', 'PG', 'ONDS', 'NFLX', 'V', 'ADBE', 'MS', 'CXW',
    'BA', 'BABA', 'DAL', 'JPM', 'C', 'AMZN', 'UAL', 'BRK.B', 'IBM', 'MSFT', 'KO',
    'MSTR', 'COIN', 'GLD', 'META', 'AAL', 'OSCR', 'QSI', 'SPY', 'NVO', 'DIS',
    'AAPL', 'BMNR', 'GRAB', 'RBLX', 'AFRM', 'NVDA', 'FUBO', 'PYPL', 'GOOG', 'BTC',
    'RGTI', 'GPRO', 'OKLO', 'AMD', 'XPEV', 'SHEL', 'TSM', 'SNOW', 'NET', 'SES',
    'TSLA', 'BP', 'UBER', 'INTC', 'MRNA', 'IREN', 'ORCL', 'HIMS', 'NBIS'
]
contracts = [Stock(t, 'SMART', 'USD') for t in tickers]
ib.qualifyContracts(*contracts)


# ══════════════════════════════════════════════════════════════════════════════
# MONGO SANITIZER
# ══════════════════════════════════════════════════════════════════════════════

def sanitize_for_mongo(d):
    """Recursively convert numpy types to native Python types."""
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out[k] = sanitize_for_mongo(v)
        elif hasattr(v, 'item'):   # catches np.bool_, np.int64, np.float64, etc.
            out[k] = v.item()
        else:
            out[k] = v
    return out


# ══════════════════════════════════════════════════════════════════════════════
# INDICATORS
# ══════════════════════════════════════════════════════════════════════════════

def calc_sma(df, period):
    return df['close'].rolling(window=period).mean()

def calc_ema(df, period):
    return df['close'].ewm(span=period, adjust=False).mean()

def calc_atr(df, period=14):
    hl  = df['high'] - df['low']
    hc  = (df['high'] - df['close'].shift()).abs()
    lc  = (df['low']  - df['close'].shift()).abs()
    df['ATR'] = pd.concat([hl, hc, lc], axis=1).max(axis=1).rolling(period).mean()
    return df

def calc_rsi(df, period=14):
    d        = df['close'].diff()
    gain     = d.clip(lower=0).ewm(com=period-1, min_periods=period).mean()
    loss     = (-d.clip(upper=0)).ewm(com=period-1, min_periods=period).mean()
    df['RSI'] = 100 - 100 / (1 + gain / (loss + 1e-10))
    return df

def calc_keltner(df, ema_period=20, atr_period=14, multiplier=3.5):
    df = calc_atr(df, atr_period)
    df['KC_mid']   = df['close'].ewm(span=ema_period, adjust=False).mean()
    df['KC_upper'] = df['KC_mid'] + multiplier * df['ATR']
    df['KC_lower'] = df['KC_mid'] - multiplier * df['ATR']
    return df

def calculate_all_indicators(df):
    df = calc_keltner(df)
    df = calc_rsi(df)
    df['EMA5']   = calc_ema(df, 5)
    df['SMA200'] = calc_sma(df, 200)
    df['SMA250'] = calc_sma(df, 250)
    return df

def get_prev_day_low(df):
    """
    Lowest low of the most recently completed trading day.
    Gossett L6: sell ½ / remaining when 5-min bar closes BELOW this level.
    """
    df2          = df.copy()
    df2['d']     = pd.to_datetime(df2['date']).dt.date
    today        = df2['d'].iloc[-1]
    prior_bars   = df2[df2['d'] < today]
    if prior_bars.empty:
        return df2['low'].iloc[:-1].min()
    prev_day     = prior_bars['d'].max()
    return df2[df2['d'] == prev_day]['low'].min()


# ══════════════════════════════════════════════════════════════════════════════
# ENTRY SIGNAL — DEEP DIP BUY  (RSI + Keltner only)
# ══════════════════════════════════════════════════════════════════════════════

def detect_entry_signal(df):
    """
    TWO conditions must be true on the latest bar:

    1. RSI RECOVERY     — RSI was ≤ 30 in last 5 bars AND is now ≥ 30
    2. KELTNER CONFIRM  — low touched KC_lower recently AND close now > KC_mid

    Returns dict with 'signal': True/False + diagnostics.
    """
    if len(df) < 10:
        return {'signal': False, 'reason': 'insufficient_data'}

    row  = df.iloc[-1]
    prev = df.iloc[-2]
    look = df.iloc[-5:]

    close    = row['close']
    rsi_now  = row['RSI']
    rsi_prev = prev['RSI']
    kc_mid   = row['KC_mid']

    # ── 1. RSI recovery ───────────────────────────────────────────
    was_oversold = (look['RSI'] <= 30).any()
    if not (was_oversold and rsi_now >= 30):
        return {'signal': False,
                'reason': f'rsi_no_recovery rsi={rsi_now:.1f} was_oversold={was_oversold}'}

    # ── 2. Keltner confirmation ───────────────────────────────────
    kc_touched   = (look['low'] <= look['KC_lower']).any()
    kc_reclaimed = close > kc_mid
    if not kc_touched:
        return {'signal': False, 'reason': 'kc_lower_not_touched'}
    if not kc_reclaimed:
        return {'signal': False,
                'reason': f'kc_mid_not_reclaimed close={close:.2f} kc_mid={kc_mid:.2f}'}

    return {
        'signal':       True,
        'rsi_now':      round(float(rsi_now), 1),
        'was_oversold': bool(was_oversold),
        'cross_30':     bool(rsi_prev < 30 <= rsi_now),
        'cross_40':     bool(rsi_prev < 40 <= rsi_now),
        'close':        round(float(close), 2),
        'kc_mid':       round(float(kc_mid), 2),
        'kc_upper':     round(float(row['KC_upper']), 2),
    }


# ══════════════════════════════════════════════════════════════════════════════
# MONGO HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def create_trade_doc(symbol, entry_price, qty, signal_info, atr):
    return {
        'instrument':         symbol,
        'direction':          'long',
        'entry_price':        float(entry_price),
        'highest_price':      float(entry_price),
        'quantity':           int(qty),
        'remaining_qty':      int(qty),
        'atr_at_entry':       round(float(atr), 4),
        'emergency_stop':     round(float(entry_price) - 2.0 * float(atr), 4),
        'trailing_active_at': round(float(entry_price) + 1.5 * float(atr), 4),
        # Gossett L6 exit-state flags
        'half_sold':          False,
        'step2a_done':        False,
        'rsi_was_above_70':   False,
        'rsi_rejected_70':    False,
        'rsi_max':            0.0,
        # Signal snapshot — sanitized to remove any numpy types
        'signal_info':        sanitize_for_mongo(signal_info),
        'entry_timestamp':    datetime.now(),
        # Standard fields
        'order_id':           str(ObjectId()),
        'exit_price':         None,
        'exit_timestamp':     None,
        'reason':             None,
        'current_pnl':        0,
        'realized_pnl':       0,
        'status':             'open',
        'created_at':         datetime.now(),
        'updated_at':         datetime.now(),
    }

def db_update(trade_id, updates):
    updates['updated_at'] = datetime.now()
    trades_collection.update_one({'_id': ObjectId(trade_id)}, {'$set': updates})

def do_sell(contract, qty, symbol, entry_price, sell_price, reason, rsi, trade_id, status='closed'):
    """Execute market sell, persist exit to MongoDB, print summary."""
    ib.placeOrder(contract, MarketOrder('SELL', qty))
    pnl = (sell_price - entry_price) * qty
    db_update(trade_id, {
        'exit_price':     float(sell_price),
        'exit_timestamp': datetime.now(),
        'reason':         reason,
        'realized_pnl':   float(pnl),
        'exit_rsi':       float(rsi),
        'status':         status,
    })
    sign = '+' if pnl >= 0 else ''
    print(f"  ✅ SELL {qty}sh @ ${sell_price:.2f}  [{reason}]  PnL: {sign}${pnl:.2f}")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN LOOP
# ══════════════════════════════════════════════════════════════════════════════

async def check_and_trade():
    """
    ╔══════════════════════════════════════════════════════════════════════════╗
    ║  GOSSETT LESSON 6 — COMPLETE EXIT RULE SET                              ║
    ╠══════════════════════════════════════════════════════════════════════════╣
    ║                                                                          ║
    ║  HARD STOPS  (checked first on every bar, always active):               ║
    ║                                                                          ║
    ║    • EMERGENCY STOP  = entry price − 2 × ATR                            ║
    ║      → EXIT ALL immediately if current price ≤ emergency_stop           ║
    ║                                                                          ║
    ║    • END-OF-DAY STOP = close below SMA 200                              ║
    ║      → EXIT ALL remaining shares                                         ║
    ║                                                                          ║
    ║  TRAILING MODE  (unlocks once price rises 1.5 ATR above entry):         ║
    ║                                                                          ║
    ║    STEP 1  ─ Sell FIRST ½ of position                                   ║
    ║      Trigger: bar closes below the PREVIOUS DAY'S LOW                   ║
    ║                                                                          ║
    ║    STEP 2a ─ Sell ½ of REMAINING shares  [RSI 70 path]                 ║
    ║      Trigger: RSI closed ≥ 70 on any prior bar AND current RSI < 70    ║
    ║                                                                          ║
    ║    STEP 3  ─ Sell ALL remaining shares  [after Step 2a]                 ║
    ║      Trigger: next bar that closes below the PREVIOUS DAY'S LOW         ║
    ║                                                                          ║
    ║    STEP 2b ─ EMA 5 FALLBACK  [if RSI peaked 65–69, never hit 70]       ║
    ║      Trigger: rsi_max ≥ 65 AND rsi never reached 70                     ║
    ║               AND price closes below EMA 5                              ║
    ║      → EXIT ALL remaining shares                                         ║
    ║                                                                          ║
    ╚══════════════════════════════════════════════════════════════════════════╝
    """

    while True:
        ib.positions()   # refresh internal positions cache

        for contract in contracts:
            symbol = contract.symbol
            print(f"\n{'='*60}")
            print(f"  {symbol}  {datetime.now().strftime('%H:%M:%S')}")
            print(f"{'='*60}")

            # ── Fetch 5-min bars (3 days for SMA 200/250 warmup) ──────────
            bars = ib.reqHistoricalData(
                contract,
                endDateTime='',
                durationStr='3 D',
                barSizeSetting='5 mins',
                whatToShow='TRADES',
                useRTH=False,
                formatDate=1
            )
            if not bars:
                print(f"  No data for {symbol}")
                continue

            df            = util.df(bars)
            df.columns    = [c.lower() for c in df.columns]
            df            = calculate_all_indicators(df)

            row           = df.iloc[-1]
            prev_row      = df.iloc[-2]
            price         = float(row['close'])
            atr           = max(float(row['ATR']), 0.01)
            rsi_now       = float(row['RSI'])
            rsi_prev      = float(prev_row['RSI'])
            ema5          = float(row['EMA5'])
            sma200        = float(row['SMA200'])
            prev_day_low  = float(get_prev_day_low(df))

            print(f"  Price=${price:.2f}  RSI={rsi_now:.1f}  EMA5=${ema5:.2f}"
                  f"  SMA200=${sma200:.2f}  PrevDayLow=${prev_day_low:.2f}")

            open_trade = trades_collection.find_one({'instrument': symbol, 'status': 'open'})

            # ══════════════════════════════════════════════════════════════
            # POSITION MANAGEMENT — EXIT LOGIC
            # ══════════════════════════════════════════════════════════════

            if open_trade:
                entry_price        = float(open_trade['entry_price'])
                orig_qty           = int(open_trade['quantity'])
                rem_qty            = int(open_trade.get('remaining_qty', orig_qty))
                half_sold          = bool(open_trade.get('half_sold', False))
                step2a_done        = bool(open_trade.get('step2a_done', False))
                rsi_was_above_70   = bool(open_trade.get('rsi_was_above_70', False))
                rsi_rejected_70    = bool(open_trade.get('rsi_rejected_70', False))
                rsi_max            = float(open_trade.get('rsi_max', 0.0))
                emergency_stop     = float(open_trade['emergency_stop'])
                trailing_active_at = float(open_trade['trailing_active_at'])
                highest_price      = float(open_trade.get('highest_price', entry_price))
                trade_id           = open_trade['_id']

                profit_pct  = (price - entry_price) / entry_price
                current_pnl = (price - entry_price) * rem_qty

                # ── Update rolling state trackers ─────────────────────────
                state_upd = {'current_pnl': float(current_pnl)}

                if price > highest_price:
                    highest_price = price
                    state_upd['highest_price'] = float(highest_price)

                if rsi_now > rsi_max:
                    rsi_max = rsi_now
                    state_upd['rsi_max'] = float(rsi_max)

                # RSI crossed above 70 — flag it
                if rsi_now >= 70 and not rsi_was_above_70:
                    rsi_was_above_70 = True
                    state_upd['rsi_was_above_70'] = True
                    print(f"  📌 RSI crossed above 70 ({rsi_now:.1f}) — tracking rejection")

                # RSI rejection: was ≥70 on previous bar, now closed below 70
                if rsi_was_above_70 and not rsi_rejected_70 and rsi_prev >= 70 and rsi_now < 70:
                    rsi_rejected_70 = True
                    state_upd['rsi_rejected_70'] = True
                    print(f"  📌 RSI 70 REJECTION confirmed (prev={rsi_prev:.1f} → now={rsi_now:.1f})")

                db_update(trade_id, state_upd)

                trailing_mode = (highest_price >= trailing_active_at)

                print(f"  OPEN  entry=${entry_price:.2f}  rem={rem_qty}/{orig_qty}"
                      f"  P&L={profit_pct:+.2%}"
                      f"  trailing={'ON' if trailing_mode else 'OFF'}"
                      f"  emerg_stop=${emergency_stop:.2f}"
                      f"  trail_at=${trailing_active_at:.2f}")
                print(f"  half_sold={half_sold}  step2a={step2a_done}"
                      f"  rsi_was70={rsi_was_above_70}  rsi_rej70={rsi_rejected_70}"
                      f"  rsi_max={rsi_max:.1f}")

                # ──────────────────────────────────────────────────────────
                # HARD STOP A — Emergency: close ≤ entry - 2 ATR
                # ──────────────────────────────────────────────────────────
                if price <= emergency_stop and rem_qty > 0:
                    do_sell(contract, rem_qty, symbol, entry_price, price,
                            f'EMERGENCY_STOP_2ATR_@{emergency_stop:.2f}',
                            rsi_now, trade_id, status='closed')
                    print(f"  🚨 EMERGENCY STOP hit @ ${price:.2f}"
                          f" (entry ${entry_price:.2f} − 2×ATR {2*float(open_trade['atr_at_entry']):.2f})")
                    continue

                # ──────────────────────────────────────────────────────────
                # HARD STOP B — End-of-day: close below SMA 200
                # ──────────────────────────────────────────────────────────
                if price < sma200 and rem_qty > 0:
                    do_sell(contract, rem_qty, symbol, entry_price, price,
                            f'EOD_STOP_BELOW_SMA200_@{sma200:.2f}',
                            rsi_now, trade_id, status='closed')
                    print(f"  🚨 END-OF-DAY STOP: close ${price:.2f} < SMA200 ${sma200:.2f}")
                    continue

                # ──────────────────────────────────────────────────────────
                # TRAILING MODE EXITS
                # ──────────────────────────────────────────────────────────
                if trailing_mode and rem_qty > 0:

                    half_qty = max(1, orig_qty // 2)

                    # ── STEP 1 — Sell FIRST ½ ─────────────────────────────
                    if not half_sold and price < prev_day_low:
                        sell_qty = min(half_qty, rem_qty)
                        new_rem  = rem_qty - sell_qty
                        do_sell(contract, sell_qty, symbol, entry_price, price,
                                f'STEP1_FIRST_HALF_BELOW_PREV_DAY_LOW_{prev_day_low:.2f}',
                                rsi_now, trade_id,
                                status='partial' if new_rem > 0 else 'closed')
                        db_update(trade_id, {'half_sold': True, 'remaining_qty': int(new_rem)})
                        half_sold = True
                        rem_qty   = new_rem
                        print(f"  📉 STEP 1: sold {sell_qty}sh. Remaining: {rem_qty}")
                        if rem_qty <= 0:
                            continue

                    # ── STEP 2a — Sell ½ of REMAINING on RSI 70 rejection ─
                    if half_sold and not step2a_done and rsi_rejected_70 and rem_qty > 0:
                        sell_qty = max(1, rem_qty // 2)
                        new_rem  = rem_qty - sell_qty
                        do_sell(contract, sell_qty, symbol, entry_price, price,
                                f'STEP2a_HALF_REMAINING_RSI70_REJECTION_rsi={rsi_now:.1f}',
                                rsi_now, trade_id,
                                status='partial' if new_rem > 0 else 'closed')
                        db_update(trade_id, {'step2a_done': True, 'remaining_qty': int(new_rem)})
                        step2a_done = True
                        rem_qty     = new_rem
                        print(f"  📉 STEP 2a: sold {sell_qty}sh (RSI 70 rejection). Remaining: {rem_qty}")
                        if rem_qty <= 0:
                            continue

                    # ── STEP 3 — Sell ALL remaining after Step 2a ─────────
                    if half_sold and step2a_done and price < prev_day_low and rem_qty > 0:
                        do_sell(contract, rem_qty, symbol, entry_price, price,
                                f'STEP3_FINAL_REMAINING_BELOW_PREV_DAY_LOW_{prev_day_low:.2f}',
                                rsi_now, trade_id, status='closed')
                        print(f"  📉 STEP 3: sold all {rem_qty}sh remaining "
                              f"(close below prev day low after RSI 70 rejection)")
                        continue

                    # ── STEP 2b — EMA 5 FALLBACK ──────────────────────────
                    ema5_fallback = (rsi_max >= 65 and not rsi_was_above_70)
                    if half_sold and ema5_fallback and price < ema5 and rem_qty > 0:
                        do_sell(contract, rem_qty, symbol, entry_price, price,
                                f'STEP2b_EMA5_FALLBACK_rsi_max={rsi_max:.1f}',
                                rsi_now, trade_id, status='closed')
                        print(f"  📉 STEP 2b EMA5 fallback: sold {rem_qty}sh "
                              f"(RSI peaked {rsi_max:.1f}, never hit 70, "
                              f"close ${price:.2f} < EMA5 ${ema5:.2f})")
                        continue

                else:
                    if not trailing_mode:
                        gap = trailing_active_at - price
                        print(f"  Waiting for trailing mode: "
                              f"need +${gap:.2f} to reach ${trailing_active_at:.2f} "
                              f"(entry + 1.5×ATR)")

            # ══════════════════════════════════════════════════════════════
            # ENTRY LOGIC — DEEP DIP BUY
            # ══════════════════════════════════════════════════════════════

            else:
                today = datetime.now().date()
                if exclude_collection.find_one({'ticker': symbol, 'date': today.isoformat()}):
                    print(f"  Already traded {symbol} today — skipping.")
                    continue

                signal = detect_entry_signal(df)

                if signal['signal']:
                    order_size = 40

                    ib.placeOrder(contract, MarketOrder('BUY', order_size))

                    doc = create_trade_doc(symbol, price, order_size, signal, atr)
                    trades_collection.insert_one(doc)
                    exclude_collection.insert_one({'ticker': symbol, 'date': today.isoformat()})

                    print(f"  🚀 DEEP DIP BUY  {symbol}")
                    print(f"     Entry:            ${price:.2f}")
                    print(f"     Shares:           {order_size}  (fixed position size)")
                    print(f"     RSI:              {signal['rsi_now']}  (was_oversold={signal['was_oversold']})")
                    print(f"     KC mid:          ${signal['kc_mid']:.2f}  (reclaimed ✓)")
                    print(f"     Emergency stop:  ${price - 2*atr:.2f}  (entry − 2×ATR)")
                    print(f"     Trailing unlocks: ${price + 1.5*atr:.2f}  (entry + 1.5×ATR)")
                    print(f"     KC Upper target: ${signal['kc_upper']:.2f}")
                else:
                    print(f"  No entry — {signal.get('reason', 'n/a')}")

            await asyncio.sleep(0.5)   # brief pause between tickers

        print(f"\n{'='*60}")
        print(f"  Scan complete. Next scan in 5 minutes.")
        print(f"{'='*60}")
        await asyncio.sleep(300)


# ══════════════════════════════════════════════════════════════════════════════
# RUN
# ══════════════════════════════════════════════════════════════════════════════

try:
    ib.run(check_and_trade())
except Exception as e:
    print(f"Error: {e}")
    import traceback; traceback.print_exc()
finally:
    ib.disconnect()
    print("Disconnected from IB.")